# Agents in LangChain

## Table of Contents
1. [What is an Agent?](#what-is-an-agent)
2. [How Agents Work — The ReAct Loop](#how-agents-work)
3. [Key Components](#key-components)
4. [Creating an Agent in LangChain](#creating-an-agent)
5. [Full Working Example](#full-working-example)
6. [Agent vs Chain](#agent-vs-chain)

---
## 1. What is an Agent?

A **Chain** follows a fixed, pre-defined sequence of steps.
An **Agent** is different — it **decides dynamically** what steps to take based on the input.

```
Chain  :  Input --> Step1 --> Step2 --> Step3 --> Output   (fixed path)
Agent  :  Input --> Think --> Pick Tool --> Observe --> Think --> Answer  (dynamic)
```

The LLM acts as the **brain** — it reads the question, decides which **tool** to use,
calls it, reads the result, and repeats until it has the final answer.

> **Real-world analogy:** A chain is a recipe. An agent is a chef who decides
> which recipe to follow (or invent) based on what ingredients are available.

---
## 2. How Agents Work — The ReAct Loop

LangChain agents use the **ReAct** pattern (**Re**ason + **Act**):

```
1. THOUGHT   ->  LLM thinks about the question
2. ACTION    ->  LLM picks a tool and calls it
3. OBSERVATION -> Agent sees the tool result
4. Repeat until LLM says FINAL ANSWER
```

**Example flow for question `What is 25 * 4 + 10?`:**

```
Thought     : I need to calculate this. I'll use the calculator tool.
Action      : calculator('25 * 4 + 10')
Observation : 110
Thought     : I now have the answer.
Final Answer: 110
```

---
## 3. Key Components

| Component | Role |
|---|---|
| **LLM** | The brain — reasons and decides what to do |
| **Tools** | Functions the agent can call (search, calculator, API, etc.) |
| **AgentExecutor** | The runtime loop — runs Think-Act-Observe repeatedly |
| **Memory** *(optional)* | Stores conversation history across turns |

### What is a Tool?
A tool is just a **Python function** decorated with `@tool` so the agent knows about it:

```python
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    'Add two numbers.'          # <-- docstring tells the LLM what this tool does
    return a + b
```

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from dotenv import load_dotenv

load_dotenv(override=True)

llm = ChatAnthropic(model='claude-sonnet-4-6', max_tokens=1024)

---
## 4. Creating an Agent in LangChain

Three steps:
1. **Define tools** — functions the agent can use
2. **Create agent** — bind tools to LLM + prompt
3. **Create AgentExecutor** — the loop that runs the agent

### Step 1 — Define Tools

In [ ]:
@tool
def multiply(a: int, b: int) -> int:
    'Multiply two integers together.'
    return a * b

@tool
def add(a: int, b: int) -> int:
    'Add two integers together.'
    return a + b

@tool
def get_word_length(word: str) -> int:
    'Return the number of characters in a word.'
    return len(word)

tools = [multiply, add, get_word_length]

# Inspect what a tool looks like to the agent
print('Tool name        :', multiply.name)
print('Tool description :', multiply.description)
print('Tool args        :', multiply.args)

### Step 2 — Create Agent

The prompt needs a special `MessagesPlaceholder` for `agent_scratchpad` —
this is where the agent writes its Thought/Action/Observation notes internally.

In [ ]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt='You are a helpful assistant. Use tools when needed.'
)

print(type(agent))

### Step 3 — Wrap in AgentExecutor

In [ ]:
# LangChain >= 1.x: no AgentExecutor needed — create_agent returns a ready-to-use agent
print('Agent is ready! Type:', type(agent).__name__)

---
## 5. Full Working Example

### Example 1 — Math using tools

In [ ]:
from langchain_core.messages import HumanMessage

# Agent will decide to use 'multiply' then 'add' tools
result = agent.invoke({'messages': [HumanMessage('What is 7 multiplied by 6, then add 15 to the result?')]})
print('Final Answer:', result['messages'][-1].content)

In [ ]:

from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

result = agent.invoke({'messages': [HumanMessage('What is 7 multiplied by 6, then add 15 to the result?')]})

# Print all intermediate steps
print('=== Agent Execution Trace ===\n')
for msg in result['messages']:
    if isinstance(msg, HumanMessage):
        print(f'[USER]        {msg.content}')
    elif isinstance(msg, AIMessage):
        if msg.tool_calls:
            for tc in msg.tool_calls:
                args_str = ', '.join(f'{k}={v}' for k, v in tc['args'].items())
                print(f'[TOOL CALL]   {tc["name"]}({args_str})')
        else:
            print(f'\n[FINAL ANSWER] {msg.content}')
    elif isinstance(msg, ToolMessage):
        print(f'[TOOL RESULT] {msg.name} → {msg.content}')

### Example 2 — Word length (uses a different tool)

In [ ]:
result = agent.invoke({'messages': [HumanMessage('How many characters are in the word "LangChain"?')]})
print('Final Answer:', result['messages'][-1].content)

### Example 3 — Agent picks the right tool on its own

In [ ]:
result = agent.invoke({'messages': [HumanMessage(
    'Add 12 and 8 together, then tell me how many characters that number has when written as a word.'
)]})
print('Final Answer:', result['messages'][-1].content)

---
## 6. Agent vs Chain

| | Chain | Agent |
|---|---|---|
| **Path** | Fixed, pre-defined | Dynamic, decided at runtime |
| **Tools** | No | Yes |
| **Loops** | No | Yes (Think-Act-Observe repeats) |
| **Use when** | Steps are known in advance | Steps depend on the input |
| **Example** | Translate then summarise | Answer questions using search/calculator |

---
## Summary

```
1. Define tools      -->  @tool decorated Python functions
2. Create agent      -->  create_tool_calling_agent(llm, tools, prompt)
3. Wrap in executor  -->  AgentExecutor(agent, tools, verbose=True)
4. Run               -->  agent_executor.invoke({'input': '...'})
```

> **Key insight:** The LLM doesn't just generate text — it decides **which tool to call**
> and **what arguments to pass**, then reads the result and continues reasoning.